In [1]:
# ============================================================
# Célula 1 — Funções de coleta
# ============================================================
import pandas as pd
import requests
from io import StringIO


def baixar_tabelas(url):
    # baixa a página e pega todas as tabelas dela
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    tabelas = pd.read_html(StringIO(html))

    for tabela in tabelas:
        # algumas tabelas têm cabeçalho de 2 linhas, por exemplo:
        # ("Box office gross", "Worldwide") -> "Box office gross Worldwide"
        if isinstance(tabela.columns, pd.MultiIndex):
            novas_colunas = []
            for coluna in tabela.columns:
                partes = []
                for parte in coluna:
                    parte = str(parte)
                    # o pandas preenche cabeçalho vazio com "Unnamed", então pulo
                    # e também não repito a mesma palavra ("Title Title")
                    if "Unnamed" not in parte and parte not in partes:
                        partes.append(parte)
                novas_colunas.append(" ".join(partes))
            tabela.columns = novas_colunas

        # tabelas sem cabeçalho vêm com colunas 0, 1, 2...
        # transformo tudo em texto para não dar erro depois
        nomes = []
        for coluna in tabela.columns:
            nomes.append(str(coluna))
        tabela.columns = nomes

    return tabelas


def achar_todas(tabelas, tem, nao_tem=None):
    # procura as tabelas pelo nome das colunas
    # (pela posição não dá, porque muda quando a página é editada)
    encontradas = []
    for tabela in tabelas:
        colunas = " ".join(tabela.columns)   # todos os nomes num texto só
        if tem in colunas:
            if nao_tem is None or nao_tem not in colunas:
                encontradas.append(tabela)
    return encontradas


def mostrar(lista_tabelas, titulo):
    # mostra quantas linhas e colunas cada tabela tem
    print(f"\n=== {titulo}: {len(lista_tabelas)} tabela(s) ===")
    numero = 1
    for tabela in lista_tabelas:
        linhas = tabela.shape[0]
        colunas = tabela.shape[1]
        print(f"Tabela {numero}: {linhas} linhas, {colunas} colunas")
        print("  Colunas:", list(tabela.columns), "\n")
        numero = numero + 1

In [2]:
# ============================================================
# Célula 2 — Coleta da Marvel
# ============================================================

# --- 1. Baixar todas as tabelas da página ---
url_marvel = "https://en.wikipedia.org/wiki/List_of_films_based_on_Marvel_Comics_publications"
tabelas_marvel = baixar_tabelas(url_marvel)
print("Total de tabelas na página:", len(tabelas_marvel))

# --- 2. Procurar as tabelas de bilheteria e de críticas ---
bilheteria_marvel_lista = achar_todas(tabelas_marvel, tem="Opening weekend")
criticas_marvel_lista   = achar_todas(tabelas_marvel, tem="CinemaScore", nao_tem="Worldwide")

mostrar(bilheteria_marvel_lista, "BILHETERIA")
mostrar(criticas_marvel_lista, "CRÍTICAS")

# --- 3. Salvar o bruto (a tabela principal é sempre a primeira) ---
bilheteria_marvel = bilheteria_marvel_lista[0]
criticas_marvel   = criticas_marvel_lista[0]

bilheteria_marvel.to_csv("../data/raw/marvel_bilheteria_wiki.csv", index=False)
criticas_marvel.to_csv("../data/raw/marvel_criticas_wiki.csv", index=False)

Total de tabelas na página: 25

=== BILHETERIA: 1 tabela(s) ===
Tabela 1: 82 linhas, 8 colunas
  Colunas: ['Title', 'Distributor(s)', 'Release date (United States)', 'Bud\xadget (mil\xadlions)', 'Box office gross[33] Opening weekend (North America)', 'Box office gross[33] North America', 'Box office gross[33] Other territories', 'Box office gross[33] Worldwide'] 


=== CRÍTICAS: 1 tabela(s) ===
Tabela 1: 83 linhas, 4 colunas
  Colunas: ['Film', 'Critical Rotten Tomatoes', 'Critical Metacritic', 'Public CinemaScore'] 



In [3]:
# ============================================================
# Célula 3 — Coleta da DC
# ============================================================

# --- 1. Baixar todas as tabelas da página ---
url_dc = "https://en.wikipedia.org/wiki/List_of_films_based_on_DC_Comics_publications"
tabelas_dc = baixar_tabelas(url_dc)
print("Total de tabelas na página:", len(tabelas_dc))

# --- 2. Procurar as tabelas de bilheteria e de críticas ---
bilheteria_dc_lista = achar_todas(tabelas_dc, tem="Opening weekend")
criticas_dc_lista   = achar_todas(tabelas_dc, tem="CinemaScore", nao_tem="Worldwide")

mostrar(bilheteria_dc_lista, "BILHETERIA")
mostrar(criticas_dc_lista, "CRÍTICAS")

# --- 3. Salvar o bruto (a tabela principal é sempre a primeira) ---
# As tabelas extras de críticas (1 a 4) são dos selos Vertigo, Paradox Press,
# WildStorm e Mad, que ficam de fora do projeto. Por isso não são salvas.
bilheteria_dc = bilheteria_dc_lista[0]
criticas_dc   = criticas_dc_lista[0]

bilheteria_dc.to_csv("../data/raw/dc_bilheteria_wiki.csv", index=False)
criticas_dc.to_csv("../data/raw/dc_criticas_wiki.csv", index=False)


Total de tabelas na página: 28

=== BILHETERIA: 1 tabela(s) ===
Tabela 1: 52 linhas, 7 colunas
  Colunas: ['Title', 'Release date (United States)', 'Budget (millions)', 'Box office gross[37] Opening weekend (North America)', 'Box office gross[37] North America', 'Box office gross[37] Worldwide', 'Notes'] 


=== CRÍTICAS: 5 tabela(s) ===
Tabela 1: 52 linhas, 4 colunas
  Colunas: ['Film', 'Rotten Tomatoes', 'Metacritic', 'CinemaScore[38]'] 

Tabela 2: 4 linhas, 8 colunas
  Colunas: ['Title', 'Distributor(s)', 'Release date', 'Rotten Tomatoes', 'Metacritic', 'CinemaScore[38]', 'Budget (million)', 'Box office'] 

Tabela 3: 2 linhas, 8 colunas
  Colunas: ['Title', 'Distributor(s)', 'Release date', 'Rotten Tomatoes', 'Metacritic', 'CinemaScore[38]', 'Budget (million)', 'Box office'] 

Tabela 4: 3 linhas, 8 colunas
  Colunas: ['Title', 'Distributor(s)', 'Release date', 'Rotten Tomatoes', 'Metacritic', 'CinemaScore[38]', 'Budget (million)', 'Box office'] 

Tabela 5: 1 linhas, 8 colunas
  Colun